# Large qubit chip demo: eigenmode and S-parameters

Simulate a **5.4 × 3.1 mm QPDK chip** with two qubits, readout resonators,
a shared bus, and vortex-trapping fill. Find two eigenmodes and sweep S11 and
S21 using **one shared mesh with 2 µm refinement** near the metal.
The [two-qubit layout](https://github.com/gdsfactory/gsim/blob/main/nbs/data/qpdk/two_qubit_straight_bus.pic.yml)
is a straight-bus variant of QPDK's filled test chip.

**Requirements:**

- QPDK: `uv pip install qpdk==0.4.0`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation

The model uses planar PEC metal on silicon with vacuum above and reflecting
PMC outer boundaries. Junctions are omitted, and both launcher ports have
matched 50 Ω loads, so the eigenmode Q includes port loading. Lengths are in µm; frequencies below are in GHz.

In [ ]:
from pathlib import Path

import gdsfactory as gf
import klayout.db as kdb
import pandas as pd
from qpdk import PDK
from qpdk.cells.chip import chip_edge
from qpdk.tech import (
    LAYER,
    LAYER_STACK,
    get_etch_section,
    material_properties,
    routing_strategies,
)
from qpdk.utils import apply_additive_metals, fill_magnetic_vortices

from gsim import gcloud
from gsim.common.polygon_utils import simplify_polygon
from gsim.common.stack import Layer, LayerStack
from gsim.palace import DrivenConfig, EigenmodeConfig, SolverConfig
from gsim.palace.mesh import generate_mesh, write_config
from gsim.palace.ports import configure_cpw_port, extract_ports
from gsim.viz import plot_mesh

PDK.activate()

### Settings

This large chip uses 2 µm refinement near metal and a coarser 300 µm mesh
away from it. Refinement is local; the whole chip is not meshed at 2 µm.
The narrowest conductor features are also 2 µm, so compare finer meshes
before relying on resonance frequencies or Q.

| Quantity | Count |
| --- | ---: |
| Mesh elements (tetrahedra) | 2,708,297 |
| Field DOFs, order 2 | 19,136,468 |

In [ ]:
refined_mesh_size_um = 2.0
max_mesh_size_um = 300.0
curve_tolerance_um = 0.1
chip_padding_um = 50.0
chip_edge_width_um = 100.0
port_length_um = 10.0

eigenmode_target_ghz = 5.0
num_eigenmodes = 2
driven_fmin_ghz = 7.0
driven_fmax_ghz = 10.0
driven_num_points = 10001
driven_adaptive_tol = 1e-3
driven_adaptive_max_samples = 40

### Build the chip

Add vortex-trapping fill and leave 50 µm of ground between the layout,
including the launchers, and the 100 µm-wide etched chip-edge frame.

In [ ]:
notebook_dir = Path("nbs") if Path("nbs").is_dir() else Path.cwd()
port_layout = gf.read.from_yaml(
    notebook_dir / "data/qpdk/two_qubit_straight_bus.pic.yml",
    routing_strategies=routing_strategies,
)
chip_bounds = port_layout.dbbox().enlarged(chip_padding_um + chip_edge_width_um)
filled_layout = gf.Component()
filled_layout << fill_magnetic_vortices(
    component=port_layout,
    rectangle_size=(15.0, 15.0),
    gap=70.0,
    stagger=2,
)
edge = filled_layout << chip_edge(
    size=(chip_bounds.width(), chip_bounds.height()),
    width=chip_edge_width_um,
    layer=LAYER.M1_ETCH,
)
edge.dmove((chip_bounds.left, chip_bounds.bottom))
chip = apply_additive_metals(filled_layout)
chip.plot()

### Convert the etch mask to metal

Subtract the etch mask from the chip footprint and simplify curved boundaries.
Copy the two bus ports from the launcher pads.

In [ ]:
layout = chip.kdb_cell.layout()
etch = kdb.Region(chip.kdb_cell.begin_shapes_rec(layout.layer(*LAYER.M1_ETCH))).merged()
footprint = kdb.Region(etch.bbox())
if curve_tolerance_um > 0:
    etch = kdb.Region(
        [simplify_polygon(p, tolerance=curve_tolerance_um) for p in etch.each()]
    ).merged()
metal = footprint - etch

mesh_layers = {"M1": (201, 0), "SUBSTRATE": (202, 0), "VACUUM": (203, 0)}
mesh_component = gf.Component()
for name, region in (("M1", metal), ("SUBSTRATE", footprint), ("VACUUM", footprint)):
    mesh_component.kdb_cell.shapes(
        mesh_component.kcl.layout.layer(*mesh_layers[name])
    ).insert(region)

# Keep the exported ports on the wide launcher pads.
for port_name in ("o1", "o2"):
    mesh_component.add_port(name=port_name, port=port_layout.ports[port_name])

### Define the stack and ports

Use QPDK's silicon and vacuum thicknesses, with the PEC sheet at `z=0`.
The CPW ports span the launcher gaps and remain inside the signal pads.

In [ ]:
stack = LayerStack(pdk_name="qpdk")
for name, zmin, zmax, material, layer_type in (
    (
        "SUBSTRATE",
        -LAYER_STACK.layers["Substrate"].thickness,
        0.0,
        "silicon",
        "dielectric",
    ),
    ("M1", 0.0, 0.0, "niobium", "conductor"),
    ("VACUUM", 0.0, LAYER_STACK.layers["Vacuum"].thickness, "vacuum", "dielectric"),
):
    stack.layers[name] = Layer(
        name=name,
        gds_layer=mesh_layers[name],
        zmin=zmin,
        zmax=zmax,
        thickness=zmax - zmin,
        material=material,
        layer_type=layer_type,
    )
stack.dielectrics = [
    dict(
        name=layer.name.lower(),
        zmin=layer.zmin,
        zmax=layer.zmax,
        material=layer.material,
    )
    for layer in stack.layers.values()
    if layer.layer_type == "dielectric"
]
stack.materials = {
    "silicon": {
        "type": "dielectric",
        "permittivity": material_properties["Si"]["relative_permittivity"],
        "loss_tangent": material_properties["Si"]["loss_tangent"],
    },
    "vacuum": {"type": "dielectric", "permittivity": 1.0},
    "niobium": {"type": "conductor"},
}

In [ ]:
for port_name in ("o1", "o2"):
    port = mesh_component.ports[port_name]
    configure_cpw_port(
        port,
        layer="M1",
        s_width=float(port.dwidth),
        gap_width=get_etch_section(port.info["cross_section"]).width,
        length=port_length_um,
        offset=port_length_um / 2,
        impedance=50.0,
        excited=False,
    )
ports = extract_ports(mesh_component, stack)
solver = SolverConfig(linear={"tolerance": 1e-9})
eigenmode_settings = EigenmodeConfig(
    num_modes=num_eigenmodes, target=eigenmode_target_ghz * 1e9
)

### Generate the shared mesh

Embed both port surfaces before meshing. The eigenmode and driven
configurations below reuse this mesh; changing geometry or port positions
requires rerunning this section. Inspect `P1` and `P2` before submitting jobs.

The mesh API requires one output directory. Cloud downloads use gsim's
default `sim-data-<job-name>` folders in the working directory.

Both configurations explicitly set `absorbing_boundary=False` to use the
same reflecting outer boundary conditions.

In [ ]:
mesh = generate_mesh(
    component=mesh_component,
    stack=stack,
    ports=ports,
    output_dir="palace-qpdk-large-chip",
    refined_mesh_size=refined_mesh_size_um,
    max_mesh_size=max_mesh_size_um,
    margin_x=0.0,
    margin_y=0.0,
    air_margin=0.0,
    fmax=driven_fmax_ghz * 1e9,
    planar_conductors=True,
    simulation_type="eigenmode",
    eigenmode_config=eigenmode_settings,
    numerical_config=solver,
    absorbing_boundary=False,
    write_config=False,
    verbosity=0,
)
print(f"Tetrahedral elements: {mesh.mesh_stats['tetrahedra']:,}")

In [ ]:
plot_mesh(
    mesh.mesh_path,
    show_groups=["M1", "P1", "P2"],
    interactive=False,
)

### Run the two-mode eigenmode simulation

Write a configuration for the shared mesh and run it on GDSFactory+ cloud.
Reuse a completed job when its inputs match, or upload and start a new job.
The shared mesh stays in place for the driven simulation, and downloads
use gsim's default result folders.

For an existing mesh, rerun **Settings** and this submission cell after
changing the requested mode count. The cell refreshes the count and clears
any driven excitation before writing the eigenmode configuration.

Save the field data for both eigenmodes so their spatial patterns can be inspected.

In [ ]:
eigenmode_settings.num_modes = num_eigenmodes
eigenmode_settings.save = num_eigenmodes
for port in ports:
    port.excited = False

write_config(
    mesh,
    stack,
    ports,
    simulation_type="eigenmode",
    eigenmode_config=eigenmode_settings,
    numerical_config=solver,
    absorbing_boundary=False,
)
input_hash, eigenmode_job_id = gcloud.check_cache_for_dir(mesh.output_dir, "palace")
if eigenmode_job_id is None:
    eigenmode_job_id = gcloud.upload(mesh.output_dir, "palace", input_hash=input_hash)
    gcloud.start(eigenmode_job_id)
eigenmode_results = gcloud.wait_for_results(eigenmode_job_id)

In [ ]:
eigenmodes = pd.read_csv(eigenmode_results["eig.csv"], skipinitialspace=True)
eigenmodes.columns = eigenmodes.columns.str.strip()
print(
    f"Converged eigenmodes: {len(eigenmodes)} / {eigenmode_settings.num_modes} requested"
)
eigenmodes

### Plot both eigenmodes

Rerun the eigenmode submission with field saving enabled before this step.
Plot the magnitude of the real electric field in the chip plane (`z=0`),
with one panel per saved mode. Eigenmode amplitudes are normalized by the
solver; compare spatial patterns rather than absolute amplitudes to identify
the resonators, feedline, or extended chip modes.

In [ ]:
from gsim.palace import plot_fields_2d

field_modes = eigenmodes.head(eigenmode_settings.save)
plot_fields_2d(
    eigenmode_results["eig.csv"].parent,
    field="E_real",
    scalar="magnitude",
    normal="z",
    origin=0.0,
    cycles=field_modes["m"].astype(int).tolist(),
    title=[
        f"Mode {int(row['m'])}: {row['Re{f} (GHz)']:.4f} GHz |Re(E)|"
        for _, row in field_modes.iterrows()
    ],
    interactive=False,
)

### Run the driven simulation

Sweep 7–10 GHz, excite `o1`, and leave `o2` matched to 50 Ω. Only the solver
configuration and excitation change; the mesh is reused. The adaptive sweep
builds a reduced model and evaluates the requested output grid.

A narrow resonance may need a smaller frequency span and finer sampling.
Zoom around a feature and compare results across frequency and mesh refinements.

In [ ]:
for port in ports:
    port.excited = port.name == "o1"

driven_settings = DrivenConfig(
    fmin=driven_fmin_ghz * 1e9,
    fmax=driven_fmax_ghz * 1e9,
    num_points=driven_num_points,
    adaptive_tol=driven_adaptive_tol,
    adaptive_max_samples=driven_adaptive_max_samples,
    reference_impedance=50.0,
    excitation_port="o1",
)
write_config(
    mesh,
    stack,
    ports,
    simulation_type="driven",
    driven_config=driven_settings,
    numerical_config=solver,
    absorbing_boundary=False,
)
frequency_step_khz = (driven_fmax_ghz - driven_fmin_ghz) * 1e6 / (driven_num_points - 1)
print(f"Output frequency spacing: {frequency_step_khz:g} kHz")

In [ ]:
input_hash, driven_job_id = gcloud.check_cache_for_dir(mesh.output_dir, "palace")
if driven_job_id is None:
    driven_job_id = gcloud.upload(mesh.output_dir, "palace", input_hash=input_hash)
    gcloud.start(driven_job_id)
driven_sparams = gcloud.wait_for_results(driven_job_id)

### Plot S-parameters

Plot S11 and S21 magnitude and phase. Dotted lines mark eigenfrequencies
within the driven sweep as references for the port-loaded response.

In [ ]:
sparam_figure = driven_sparams.plot_plotly()
for frequency_ghz in eigenmodes["Re{f} (GHz)"]:
    if driven_fmin_ghz <= frequency_ghz <= driven_fmax_ghz:
        sparam_figure.add_vline(
            x=float(frequency_ghz),
            line_dash="dot",
            line_color="gray",
            row="all",
            col=1,
        )
sparam_figure.update_layout(title="Large qubit chip: S11 and S21")
sparam_figure